In [1]:
import numpy as np
import ksig


def sample_ar1_timeseries(
    n_series: int,
    seq_len: int,
    phi: float,
    sigma: float,
    mean_shift: float = 0.0,
    n_feat: int = 1,
    seed: int | None = None,
) -> np.ndarray:
    """
    Sample multivariate AR(1) time series.

    Returns
    -------
    X : np.ndarray
        Shape (n_series, seq_len, n_feat)
    """
    rng = np.random.default_rng(seed)

    X = np.zeros((n_series, seq_len, n_feat), dtype=np.float64)

    # Initialize first timestep from stationary-ish scale
    init_scale = sigma / max(np.sqrt(1.0 - phi**2), 1e-8) if abs(phi) < 1 else sigma
    X[:, 0, :] = mean_shift + rng.normal(loc=0.0, scale=init_scale, size=(n_series, n_feat))

    for t in range(1, seq_len):
        noise = rng.normal(loc=0.0, scale=sigma, size=(n_series, n_feat))
        X[:, t, :] = mean_shift + phi * (X[:, t - 1, :] - mean_shift) + noise

    return X


def add_time_channel(X: np.ndarray) -> np.ndarray:
    """
    Optionally append normalized time as an extra channel.
    This can help if you want sensitivity to parametrization / timing.
    """
    n, t, _ = X.shape
    time = np.linspace(0.0, 1.0, t, dtype=X.dtype)[None, :, None]
    time = np.repeat(time, n, axis=0)
    return np.concatenate([X, time], axis=-1)


def mmd2_biased(Kxx: np.ndarray, Kyy: np.ndarray, Kxy: np.ndarray) -> float:
    """
    Biased estimator of MMD^2:
        E[k(X,X')] + E[k(Y,Y')] - 2 E[k(X,Y)]
    where diagonals are included.
    """
    return float(Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean())


def mmd2_unbiased(Kxx: np.ndarray, Kyy: np.ndarray, Kxy: np.ndarray) -> float:
    """
    Unbiased estimator of MMD^2:
        1/(m(m-1)) sum_{i!=j} Kxx_ij
      + 1/(n(n-1)) sum_{i!=j} Kyy_ij
      - 2/(mn)      sum_{i,j}  Kxy_ij
    """
    m = Kxx.shape[0]
    n = Kyy.shape[0]

    if m < 2 or n < 2:
        raise ValueError("Need at least 2 samples per group for unbiased MMD.")

    sum_xx = (Kxx.sum() - np.trace(Kxx)) / (m * (m - 1))
    sum_yy = (Kyy.sum() - np.trace(Kyy)) / (n * (n - 1))
    sum_xy = Kxy.mean()

    return float(sum_xx + sum_yy - 2.0 * sum_xy)


def main():
    # -----------------------------
    # 1) Sample two time-series datasets
    # -----------------------------
    n_x = 64
    n_y = 64
    seq_len = 80
    n_feat = 1

    # Source X: AR(1) with stronger persistence
    X = sample_ar1_timeseries(
        n_series=n_x,
        seq_len=seq_len,
        phi=0.85,
        sigma=0.25,
        mean_shift=0.0,
        n_feat=n_feat,
        seed=0,
    )

    # Source Y: different AR(1) dynamics + slight mean shift
    Y = sample_ar1_timeseries(
        n_series=n_y,
        seq_len=seq_len,
        phi=0.55,
        sigma=0.25,
        mean_shift=0.4,
        n_feat=n_feat,
        seed=1,
    )

    # Optional: add time as extra channel
    # X = add_time_channel(X)
    # Y = add_time_channel(Y)

    # -----------------------------
    # 2) Build the KSig kernel
    # -----------------------------
    n_levels = 4
    static_kernel = ksig.static.kernels.RBFKernel()
    sig_kernel = ksig.kernels.SignatureKernel(
        n_levels=n_levels,
        static_kernel=static_kernel,
    )

    # -----------------------------
    # 3) Compute Gram matrices
    # -----------------------------
    Kxx = np.asarray(sig_kernel(X))
    Kyy = np.asarray(sig_kernel(Y))
    Kxy = np.asarray(sig_kernel(X, Y))

    # -----------------------------
    # 4) Compute MMD^2
    # -----------------------------
    mmd2_b = mmd2_biased(Kxx, Kyy, Kxy)
    mmd2_u = mmd2_unbiased(Kxx, Kyy, Kxy)

    print(f"Kxx shape: {Kxx.shape}")
    print(f"Kyy shape: {Kyy.shape}")
    print(f"Kxy shape: {Kxy.shape}")
    print(f"Biased   MMD^2: {mmd2_b:.8f}")
    print(f"Unbiased MMD^2: {mmd2_u:.8f}")


if __name__ == "__main__":
    main()

Kxx shape: (64, 64)
Kyy shape: (64, 64)
Kxy shape: (64, 64)
Biased   MMD^2: 0.22038328
Unbiased MMD^2: 0.20721129
